# Ky thuat 1: CLAHE (Contrast Limited Adaptive Histogram Equalization)

Notebook nay thuc hien cac thi nghiem minh hoa cho Ky thuat 1 theo dung yeu cau Muc 2.6 trong de tai.


### Co dung AI ho tro
- **Cong cu**: Claude 3.5 Sonnet / Gemini 2.5 Pro
- **Muc dich**: Thiet lap cau hinh hien thi va import ham tu cai dat tu `src.clahe`.
- **Prompt**:
> Hay import cac thu vien NumPy, Matplotlib va cac ham tu viet trong `src.clahe` de doc anh thu nghiem tu `data/sample_uneven_light.png`.
- **Da sua lai nhung gi**: Dieu chinh duong dan thu muc `../` de import chinh xac tu thu muc `src`.


In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from src.clahe import clip_histogram, build_tile_luts, clahe

# Doc anh mau
img = plt.imread('../data/sample_uneven_light.png')
if img.ndim == 3:
    img = (img[:, :, 0] * 255).astype(np.uint8)
else:
    img = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Anh goc (Anh sang khong deu + Nhieu)')
plt.axis('off')
plt.show()


## (a) Anh huong cua tham so: Quet clip_limit va grid

### Co dung AI ho tro
- **Cong cu**: Claude 3.5 Sonnet
- **Muc dich**: Dung luoi anh 4x4 quet 4 gia tri grid va 4 gia tri clip_limit.
- **Prompt**:
> Viet doan code quet grid in [(2,2), (4,4), (8,8), (16,16)] va clip_limit in [1.0, 2.0, 5.0, 20.0] dung ham clahe tu viet va hien thi duoi dang ma tran anh 4x4 bang Matplotlib.
- **Da sua lai nhung gi**: Can chinh tieu de va layout hien thi.


In [ ]:
grids = [(2, 2), (4, 4), (8, 8), (16, 16)]
clip_limits = [1.0, 2.0, 5.0, 20.0]

fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for i, g in enumerate(grids):
    for j, cl in enumerate(clip_limits):
        out = clahe(img, grid=g, clip_limit=cl)
        axes[i, j].imshow(out, cmap='gray')
        axes[i, j].set_title(f"Grid: {g}\nClip: {cl}")
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()


## (b) So sanh voi thuat toan da hoc

### Co dung AI ho tro
- **Cong cu**: Gemini 2.5 Pro
- **Muc dich**: Cai dat doi sanh 4 phuong phap: Keo gian tuyen tinh, Gamma correction, HE toan cuc va CLAHE kem bieu do Histogram.
- **Prompt**:
> Viet code so sanh 4 ky thuat tang cuong tuong phan tren cung 1 anh: Linear stretching, Gamma (0.5), Global Histogram Equalization, va CLAHE; hien thi anh ket qua va bieu do histogram cua tung anh.
- **Da sua lai nhung gi**: Viet truc tiep ham tinh HE toan cuc bang CDF chuan.


In [ ]:
# 1. Keo gian tuong phan tuyen tinh
min_val, max_val = np.min(img), np.max(img)
linear_stretched = ((img - min_val) / (max_val - min_val) * 255.0).astype(np.uint8)

# 2. Gamma correction (gamma = 0.5)
gamma = 0.5
gamma_corrected = np.clip(255.0 * (img / 255.0) ** gamma, 0, 255).astype(np.uint8)

# 3. Can bang luoc do xam toan cuc (Global HE)
hist = np.bincount(img.ravel(), minlength=256)
cdf = np.cumsum(hist)
lut_he = np.round(255.0 * cdf / cdf[-1]).astype(np.uint8)
global_he = lut_he[img]

# 4. CLAHE tu viet
res_clahe = clahe(img, grid=(8, 8), clip_limit=2.0)

methods = [
    ('Keo gian tuyen tinh', linear_stretched),
    ('Hieu chinh Gamma (0.5)', gamma_corrected),
    ('Can bang HE toan cuc', global_he),
    ('CLAHE (8x8, clip=2.0)', res_clahe)
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, (name, res) in enumerate(methods):
    axes[0, col].imshow(res, cmap='gray')
    axes[0, col].set_title(name)
    axes[0, col].axis('off')
    
    axes[1, col].hist(res.ravel(), bins=256, range=[0, 256], color='black', alpha=0.7)
    axes[1, col].set_title(f"Histogram: {name}")
    axes[1, col].set_xlim([0, 256])

plt.tight_layout()
plt.show()
